## **Supervised Model III – XGBoost**
from kamiya's XGBoost_v3c, its basically the same training, but stricter on data leakage within the cv training (to prevent over optimistic results, amongst other benefits i forget)

In [1]:
results = []

In [2]:
import matplotlib.pyplot as plt
import zipfile, warnings
import xgboost as xgb
import pandas as pd
import numpy as np
import optuna
from scipy.stats import entropy
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score


TOTAL_ITEMS = 1000
RATING_RANGE = range(6)

/Users/tori/Documents/OFFICES/SCHOOL/Y3S2/Machine_Learning_421_SMU/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### **Data Loading**

In [3]:
# Helper functions

def load_npz(path: str) -> tuple[pd.DataFrame, pd.DataFrame | None]:
    data = np.load(path)
    XX = pd.DataFrame(data["X"], columns=["user", "item", "rating"])
    yy = None
    if "y" in data:
        yy = pd.DataFrame(data["y"], columns=["user", "label"])
    return XX, yy


def combine_labeled_data(
    *npz_paths: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Merge multiple labeled .npz files into combined DataFrames.

    If a user appears in more than one file, their label is taken 
    from the first file that contains them.

    Returns:
        XX: interaction DataFrame (user, item, rating)
        yy: label DataFrame (user, label)
    """
    all_X: list[pd.DataFrame] = []
    all_y: list[pd.DataFrame] = []

    for path in npz_paths:
        data = np.load(path)
        all_X.append(pd.DataFrame(data["X"], columns=["user", "item", "rating"]))
        all_y.append(pd.DataFrame(data["y"], columns=["user", "label"]))

    XX = pd.concat(all_X, ignore_index=True)
    yy = pd.concat(all_y, ignore_index=True).drop_duplicates(
        subset="user", keep="first"
    )

    n_anom = int(yy["label"].sum())
    print(
        f"Combined {len(npz_paths)} files\n"
        f"{yy.shape[0]} users ({n_anom} anomalous, {yy.shape[0] - n_anom} normal), "
        f"{XX.shape[0]} interactions"
    )

    return XX, yy


def compute_item_stats(XX_train: pd.DataFrame) -> dict:
    """Compute item-level statistics from training interactions ONLY.

    Pass the returned dict to build_features() for both train and test
    to prevent data leakage.
    """
    item_avg = XX_train.groupby("item")["rating"].mean().rename("item_avg_rating")
    item_pop = XX_train.groupby("item")["user"].count().rename("item_popularity")
    return {"item_avg_rating": item_avg, "item_popularity": item_pop}


def build_features(
    XX: pd.DataFrame,
    item_stats: dict,
    total_items: int=TOTAL_ITEMS,
) -> pd.DataFrame:
    """Build user-level features from raw interactions.

    Returns DataFrame with a 'user' column and all engineered features.
    """
    item_avg = item_stats["item_avg_rating"]
    item_pop = item_stats["item_popularity"]

    # Basic rating statistics
    stats = XX.groupby("user")["rating"].agg(
        rating_mean="mean",
        rating_std="std",
        rating_median="median",
        rating_min="min",
        rating_max="max",
        rating_count="count",
    )
    stats["rating_std"] = stats["rating_std"].fillna(0)
    stats["rating_range"] = stats["rating_max"] - stats["rating_min"]

    # Rating proportions + entropy
    rdist = XX.groupby(["user", "rating"]).size().unstack(fill_value=0)
    rdist = rdist.reindex(columns=RATING_RANGE, fill_value=0)
    rprops = rdist.div(rdist.sum(axis=1), axis=0)
    rprops.columns = [f"prop_rating_{i}" for i in RATING_RANGE]

    stats["rating_entropy"] = rprops.apply(
        lambda row: entropy(row.values[row.values > 0]), axis=1
    )
    stats = stats.join(rprops)

    # Extreme rating proportions
    stats["prop_extreme"] = rprops["prop_rating_0"] + rprops["prop_rating_5"]

    # Item coverage
    stats["unique_items_rated"] = XX.groupby("user")["item"].nunique()
    stats["item_coverage_ratio"] = stats["unique_items_rated"] / total_items

    # Item popularity (frozen from training)
    XX_pop = XX.merge(item_pop, left_on="item", right_index=True, how="left")
    XX_pop["item_popularity"] = XX_pop["item_popularity"].fillna(0)
    pop_f = XX_pop.groupby("user")["item_popularity"].agg(
        avg_item_popularity="mean",
        std_item_popularity="std",
    )
    pop_f["std_item_popularity"] = pop_f["std_item_popularity"].fillna(0)
    stats = stats.join(pop_f)

    # Deviation from item average (frozen from training)
    XX_dev = XX.merge(item_avg, left_on="item", right_index=True, how="left")
    global_train_mean = item_avg.mean()
    XX_dev["item_avg_rating"] = XX_dev["item_avg_rating"].fillna(global_train_mean)
    XX_dev["deviation"] = XX_dev["rating"] - XX_dev["item_avg_rating"]

    dev_f = XX_dev.groupby("user")["deviation"].agg(
        mean_deviation="mean",
        std_deviation="std",
        abs_mean_deviation=lambda x: np.mean(np.abs(x)),
    )
    dev_f["std_deviation"] = dev_f["std_deviation"].fillna(0)
    stats = stats.join(dev_f)

    # Average quality of items targeted (frozen from training)
    iqf = XX_dev.groupby("user")["item_avg_rating"].agg(
        avg_item_avg_rating="mean",
        std_item_avg_rating="std",
    )
    iqf["std_item_avg_rating"] = iqf["std_item_avg_rating"].fillna(0)
    stats = stats.join(iqf)

    return stats.reset_index()


def make_fold_features(XX_raw, yy_raw, train_users, val_users):
    XX_tr = XX_raw[XX_raw['user'].isin(train_users)].copy()
    XX_val = XX_raw[XX_raw['user'].isin(val_users)].copy()

    yy_tr = yy_raw[yy_raw['user'].isin(train_users)].copy()
    yy_val = yy_raw[yy_raw['user'].isin(val_users)].copy()

    item_stats_tr = compute_item_stats(XX_tr)

    feats_tr = build_features(XX_tr, item_stats=item_stats_tr).merge(yy_tr, on='user')
    feats_val = build_features(XX_val, item_stats=item_stats_tr).merge(yy_val, on='user')

    feature_cols_fold = [c for c in feats_tr.columns if c not in ['user', 'label']]
    feats_val = feats_val[['user', 'label'] + feature_cols_fold]

    scaler_fold = RobustScaler()
    X_tr = scaler_fold.fit_transform(feats_tr[feature_cols_fold].values)
    X_val = scaler_fold.transform(feats_val[feature_cols_fold].values)

    y_tr = feats_tr['label'].values
    y_val = feats_val['label'].values

    return X_tr, y_tr, X_val, y_val, item_stats_tr, feature_cols_fold, scaler_fold


def codabench_metrics(
    test_labels: np.ndarray,
    scores: np.ndarray,
    model_name: str,
    verbose=False,
) -> dict:
    """Compute metrics at the fixed Codabench threshold of 0.5.

    Use this on calibrated scores to see exactly what Codabench will report.
    """
    test_labels = np.asarray(test_labels).astype(int)
    scores = np.asarray(scores).astype(float)

    preds = (scores >= 0.5).astype(int)
    metrics = {
        "model": model_name,
        "AUC": roc_auc_score(test_labels, scores),
        "Precision": precision_score(test_labels, preds, zero_division=0),
        "Recall": recall_score(test_labels, preds, zero_division=0),
        "F1": f1_score(test_labels, preds, zero_division=0),
        "threshold": 0.5,
    }

    if verbose: 
        print(f"{model_name} (Codabench t=0.5)")
        print(f"# AUC:       {metrics['AUC']:.4f}")
        print(f"# Precision: {metrics['Precision']:.4f}")
        print(f"# Recall:    {metrics['Recall']:.4f}")
        print(f"# F1 Score:  {metrics['F1']:.4f}")

    return metrics


In [4]:
# Phase 3 data loading
XX_all, yy_all = combine_labeled_data(
    "data/training_batch_with_labels.npz", 
    "data/first_batch_with_labels.npz", 
    "data/second_batch_with_labels.npz"
)

Combined 3 files
3060 users (260 anomalous, 2800 normal), 479433 interactions


In [5]:
# Build features on all training data
item_stats_full = compute_item_stats(XX_all)
full_train_df = build_features(XX_all, item_stats_full).merge(yy_all, on="user")
feature_cols = [c for c in full_train_df.columns if c not in ['user', 'label']]

X_trainval = full_train_df[feature_cols].values
y_trainval = full_train_df["label"].values

scaler = RobustScaler()
X_trainval_s = scaler.fit_transform(X_trainval)

print(f'Training users: {len(y_trainval)}')
print(f'Features:       {len(feature_cols)}')


Training users: 3060
Features:       24


#### **Optuna Hyperparameter Search**

In [6]:
spw_global = np.sum(y_trainval == 0) / np.sum(y_trainval == 1)

def objective(trial):
    # Choose grow policy first, as it affects which depth param to use
    grow_policy = trial.suggest_categorical('grow_policy', ['depthwise', 'lossguide'])

    params = dict(
        n_estimators       = trial.suggest_int('n_estimators', 100, 4000),
        learning_rate      = trial.suggest_float('learning_rate', 0.001, 0.15, log=True),
        subsample          = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree   = trial.suggest_float('colsample_bytree', 0.4, 1.0),
        colsample_bylevel  = trial.suggest_float('colsample_bylevel', 0.4, 1.0),
        colsample_bynode   = trial.suggest_float('colsample_bynode', 0.4, 1.0),
        min_child_weight   = trial.suggest_int('min_child_weight', 1, 10),
        gamma              = trial.suggest_float('gamma', 0.0, 2.0),
        reg_alpha          = trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        reg_lambda         = trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        max_delta_step     = trial.suggest_int('max_delta_step', 0, 10),
        scale_pos_weight   = trial.suggest_float('scale_pos_weight', spw_global * 0.5, spw_global * 3.0),
        grow_policy        = grow_policy,
    )

    # lossguide uses max_leaves; depthwise uses max_depth
    if grow_policy == 'lossguide':
        params['max_leaves'] = trial.suggest_int('max_leaves', 16, 512)
    else:
        params['max_depth'] = trial.suggest_int('max_depth', 3, 12)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=trial.number)
    aucs = []
    for tr_i, val_i in cv.split(X_trainval_s, y_trainval):
        m = xgb.XGBClassifier(
            **params, eval_metric='aucpr', early_stopping_rounds=50,
            random_state=42, n_jobs=-1, tree_method='hist'
        )
        m.fit(X_trainval_s[tr_i], y_trainval[tr_i],
              eval_set=[(X_trainval_s[val_i], y_trainval[val_i])], verbose=False)
        aucs.append(roc_auc_score(y_trainval[val_i],
                                   m.predict_proba(X_trainval_s[val_i])[:, 1]))
    return np.mean(aucs)

In [7]:
# Search for the best hyperparameters using all training data (may be slightly optimistic)
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=300, show_progress_bar=True)

best_params = study.best_params
print(f'Best CV AUC: {study.best_value:.4f}')
print('Best params:', best_params)

[I 2026-03-28 23:58:22,820] A new study created in memory with name: no-name-3f411d39-c532-4f55-951f-9d9740ad5e77
Best trial: 0. Best value: 0.903736:   0%|          | 1/300 [00:01<09:14,  1.85s/it]

[I 2026-03-28 23:58:24,680] Trial 0 finished with value: 0.9037362637362637 and parameters: {'grow_policy': 'depthwise', 'n_estimators': 1507, 'learning_rate': 0.05772797731257423, 'subsample': 0.7533702630424091, 'colsample_bytree': 0.999506464128522, 'colsample_bylevel': 0.6855171594305399, 'colsample_bynode': 0.9313806177586307, 'min_child_weight': 6, 'gamma': 1.7683971004749905, 'reg_alpha': 0.4828791508298806, 'reg_lambda': 5.525445695087913, 'max_delta_step': 10, 'scale_pos_weight': 30.429563738343575, 'max_depth': 9}. Best is trial 0 with value: 0.9037362637362637.


Best trial: 1. Best value: 0.904141:   1%|          | 2/300 [00:05<15:37,  3.15s/it]

[I 2026-03-28 23:58:28,730] Trial 1 finished with value: 0.9041414835164836 and parameters: {'grow_policy': 'lossguide', 'n_estimators': 471, 'learning_rate': 0.02192699913085882, 'subsample': 0.7108818020384409, 'colsample_bytree': 0.48920060330015075, 'colsample_bylevel': 0.8397942185324399, 'colsample_bynode': 0.5094711331322134, 'min_child_weight': 6, 'gamma': 1.595307338238902, 'reg_alpha': 0.044686949989653287, 'reg_lambda': 0.000661413277121812, 'max_delta_step': 4, 'scale_pos_weight': 20.41699604594669, 'max_leaves': 371}. Best is trial 1 with value: 0.9041414835164836.


Best trial: 1. Best value: 0.904141:   1%|          | 3/300 [00:08<13:20,  2.70s/it]

...

[I 2026-03-29 00:13:15,380] Trial 297 finished with value: 0.9033447802197803 and parameters: {'grow_policy': 'lossguide', 'n_estimators': 2526, 'learning_rate': 0.04155477016215732, 'subsample': 0.7572725459361689, 'colsample_bytree': 0.9329579909520491, 'colsample_bylevel': 0.612110729119083, 'colsample_bynode': 0.984364811193112, 'min_child_weight': 4, 'gamma': 1.6209418750935096, 'reg_alpha': 0.0060447727658970996, 'reg_lambda': 0.01732422670230234, 'max_delta_step': 2, 'scale_pos_weight': 30.956531217104082, 'max_leaves': 208}. Best is trial 161 with value: 0.9199072802197803.


Best trial: 161. Best value: 0.919907: 100%|█████████▉| 299/300 [14:57<00:04,  4.05s/it]

[I 2026-03-29 00:13:20,257] Trial 298 finished with value: 0.9133997252747253 and parameters: {'grow_policy': 'lossguide', 'n_estimators': 2852, 'learning_rate': 0.05835306903814422, 'subsample': 0.7745433671945967, 'colsample_bytree': 0.8908792941999746, 'colsample_bylevel': 0.5630414947706404, 'colsample_bynode': 0.7869859764975052, 'min_child_weight': 1, 'gamma': 1.0749542413431452, 'reg_alpha': 0.0003246344035810784, 'reg_lambda': 0.03600239311542598, 'max_delta_step': 1, 'scale_pos_weight': 27.528669642358395, 'max_leaves': 153}. Best is trial 161 with value: 0.9199072802197803.


Best trial: 161. Best value: 0.919907: 100%|██████████| 300/300 [14:59<00:00,  3.00s/it]

[I 2026-03-29 00:13:22,617] Trial 299 finished with value: 0.8987156593406593 and parameters: {'grow_policy': 'lossguide', 'n_estimators': 2902, 'learning_rate': 0.12444587686014223, 'subsample': 0.6531159425950233, 'colsample_bytree': 0.8247226341913549, 'colsample_bylevel': 0.5667066392119727, 'colsample_bynode': 0.7698605164818512, 'min_child_weight': 1, 'gamma': 1.067760567674595, 'reg_alpha': 0.0003243834876202583, 'reg_lambda': 5.506956621277323, 'max_delta_step': 1, 'scale_pos_weight': 27.47763015454404, 'max_leaves': 153}. Best is trial 161 with value: 0.9199072802197803.
Best CV AUC: 0.9199
Best params: {'grow_policy': 'depthwise', 'n_estimators': 2816, 'learning_rate': 0.04847419447256601, 'subsample': 0.9022548031690737, 'colsample_bytree': 0.9028524273943843, 'colsample_bylevel': 0.6250673358447384, 'colsample_bynode': 0.8052569875187007, 'min_child_weight': 4, 'gamma': 0.9974162854090418, 'reg_alpha': 0.0004368646931066822, 'reg_lambda': 0.04298410574008765, 'max_delta_ste

#### **Cross-Validation using Best Parameters**

In [8]:
cv = StratifiedKFold(n_splits=20, shuffle=True, random_state=42)

oof_scores = np.zeros(len(y_trainval))
fold_aucs = []
fold_artifacts = []

for fold, (tr_idx, va_idx) in enumerate(cv.split(X_trainval, y_trainval), 1):
    # user-level split reference
    train_users = full_train_df.iloc[tr_idx]["user"].values
    val_users   = full_train_df.iloc[va_idx]["user"].values

    # build fold-specific train/val features with fold-specific frozen stats
    X_tr, y_tr, X_va, y_va, item_stats_tr, feature_cols_fold, scaler_fold = make_fold_features(
        XX_raw=XX_all,
        yy_raw=yy_all,
        train_users=train_users,
        val_users=val_users
    )

    model = xgb.XGBClassifier(
        **best_params,
        eval_metric="auc",
        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    )

    model.fit(X_tr, y_tr, verbose=False)

    p_va = model.predict_proba(X_va)[:, 1]
    oof_scores[va_idx] = p_va

    fold_auc = roc_auc_score(y_va, p_va)
    fold_aucs.append(fold_auc)

    fold_artifacts.append({
        "model": model,
        "item_stats": item_stats_tr,
        "feature_cols": feature_cols_fold,
        "scaler": scaler_fold,
    })

    print(f"Fold {fold:02d} AUC: {fold_auc:.4f}")

print(f"\nMean fold AUC: {np.mean(fold_aucs):.4f}")
print(f"OOF AUC:       {roc_auc_score(y_trainval, oof_scores):.4f}")

Fold 01 AUC: 0.8791
Fold 02 AUC: 0.9077
Fold 03 AUC: 0.9066
Fold 04 AUC: 0.9297
Fold 05 AUC: 0.8995
Fold 06 AUC: 0.9819
Fold 07 AUC: 0.9324
Fold 08 AUC: 0.8764
Fold 09 AUC: 0.9297
Fold 10 AUC: 0.8846
Fold 11 AUC: 0.9159
Fold 12 AUC: 0.8495
Fold 13 AUC: 0.9280
Fold 14 AUC: 0.9538
Fold 15 AUC: 0.9181
Fold 16 AUC: 0.9621
Fold 17 AUC: 0.8808
Fold 18 AUC: 0.8236
Fold 19 AUC: 0.9324
Fold 20 AUC: 0.9725

Mean fold AUC: 0.9132
OOF AUC:       0.9112


#### **Results Calibration**

In [9]:
# Calibrate OOF probabilities using sigmoid calibration for better precision/recall
calibrator = LogisticRegression(max_iter=1000, random_state=42)
calibrator.fit(oof_scores.reshape(-1, 1), y_trainval)

oof_scores_cal = calibrator.predict_proba(oof_scores.reshape(-1, 1))[:, 1]
print(f"Calibrated OOF AUC: {roc_auc_score(y_trainval, oof_scores_cal):.4f}")

Calibrated OOF AUC: 0.9112


In [10]:
# Threshold analysis on calibrated OOF predictions
thresholds = np.linspace(0.01, 0.99, 500)
j_scores, f1_scores_t, prec_list, rec_list = [], [], [], []

for t in thresholds:
    preds_t = (oof_scores_cal >= t).astype(int)

    tp = np.sum((preds_t == 1) & (y_trainval == 1))
    tn = np.sum((preds_t == 0) & (y_trainval == 0))
    fp = np.sum((preds_t == 1) & (y_trainval == 0))
    fn = np.sum((preds_t == 0) & (y_trainval == 1))

    sens = tp / (tp + fn + 1e-9)
    spec = tn / (tn + fp + 1e-9)
    prec = tp / (tp + fp + 1e-9)
    f1_t = 2 * prec * sens / (prec + sens + 1e-9)

    j_scores.append(sens + spec - 1)
    f1_scores_t.append(f1_t)
    prec_list.append(prec)
    rec_list.append(sens)

best_j_idx  = np.argmax(j_scores)
best_f1_idx = np.argmax(f1_scores_t)

print(f"Optimal threshold — Youden's J : {thresholds[best_j_idx]:.3f}  "
      f"(prec={prec_list[best_j_idx]:.3f}, rec={rec_list[best_j_idx]:.3f})")
print(f"Optimal threshold — Best F1   : {thresholds[best_f1_idx]:.3f}  "
      f"(prec={prec_list[best_f1_idx]:.3f}, rec={rec_list[best_f1_idx]:.3f})")

Optimal threshold — Youden's J : 0.043  (prec=0.338, rec=0.808)
Optimal threshold — Best F1   : 0.513  (prec=0.792, rec=0.527)


#### **Model Predictions**

In [11]:
# Load up test data
XX_test, _ = load_npz("data/third_batch.npz")

In [12]:
# Build fold-specific ensemble predictions on test
fold_test_preds = []

for fold_id, art in enumerate(fold_artifacts, 1):
    model = art["model"]
    item_stats_fold = art["item_stats"]
    feature_cols_fold = art["feature_cols"]
    scaler_fold = art["scaler"]

    # Build test features using this fold's item stats
    test_df_fold = build_features(XX_test, item_stats_fold)

    # Ensure all required columns exist
    for c in feature_cols_fold:
        if c not in test_df_fold.columns:
            test_df_fold[c] = 0.0

    X_test_fold = test_df_fold[feature_cols_fold].values
    X_test_fold_s = scaler_fold.transform(X_test_fold)

    p_test_fold = model.predict_proba(X_test_fold_s)[:, 1]
    fold_test_preds.append(p_test_fold)

    print(f"Generated test predictions from fold {fold_id:02d}")

fold_test_preds = np.column_stack(fold_test_preds)
y_score_raw = fold_test_preds.mean(axis=1)

# Calibrate averaged fold-ensemble scores
y_score_cal = calibrator.predict_proba(y_score_raw.reshape(-1, 1))[:, 1]

# Normalise to [0,1]
y_score_norm = (y_score_cal - y_score_cal.min()) / (y_score_cal.max() - y_score_cal.min() + 1e-9)

print("Test prediction shape:", y_score_norm.shape)
print("Prediction range:", y_score_norm.min(), y_score_norm.max())

Generated test predictions from fold 01
Generated test predictions from fold 02
Generated test predictions from fold 03
Generated test predictions from fold 04
Generated test predictions from fold 05
Generated test predictions from fold 06
Generated test predictions from fold 07
Generated test predictions from fold 08
Generated test predictions from fold 09
Generated test predictions from fold 10
Generated test predictions from fold 11
Generated test predictions from fold 12
Generated test predictions from fold 13
Generated test predictions from fold 14
Generated test predictions from fold 15
Generated test predictions from fold 16
Generated test predictions from fold 17
Generated test predictions from fold 18
Generated test predictions from fold 19
Generated test predictions from fold 20
Test prediction shape: (1625,)
Prediction range: 0.0 0.9999999987563004


#### **Evaluation (local/Codabench)**

**Phase 3 Codabench Results**

In [13]:
# Save submission
np.savez('submission.npz', predictions=y_score_norm)
with zipfile.ZipFile('submission.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write('submission.npz', arcname='submission.npz')
pd.DataFrame({'predictions': y_score_norm}).to_csv('submission.csv', index=False)
print('submission.zip ready for Codabench')

submission.zip ready for Codabench


# **XGBoost_test.zip**
AUC:       0.9116
Precision: 0.6610
Recall:    0.3120
F1 Score:  0.4239

**Phase 2 local results**

In [ ]:
# # Final alignment by user just to be sure
# test_users = build_features(XX_test, fold_artifacts[0]["item_stats"])[["user"]]
# yy_test_aligned = test_users.merge(yy_test, on="user", how="left")

# # Evaluate using Codabench threshold = 0.5
# codabench_metrics(yy_test_aligned["label"].values, y_score_norm, "XGBoost", verbose=True)

# **XGBoost (Codabench t=0.5) Phase 2**
AUC:       0.6373
Precision: 0.3043
Recall:    0.1167
F1 Score:  0.1687

{'model': 'XGBoost',
 'AUC': 0.6372916666666667,
 'Precision': 0.30434782608695654,
 'Recall': 0.11666666666666667,
 'F1': 0.1686746987951807,
 'threshold': 0.5}